In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### **Data Reading**

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@tjdatabricksete.dfs.core.windows.net/customer")

In [0]:
display(df)

In [0]:
df = df.drop("_rescued_data")


In [0]:
#Extracting the domain name form the email column 
df = df.withColumn("domain", split(col("email"),"@")[1])
display(df)

In [0]:
df.groupBy("domain").agg(count("customer_id").alias("total_customers")).sort("total_customers", ascending=False).display()

In [0]:
df = df.withColumn("fullname", concat(col("first_name"), lit(" "), col("last_name")))\
        .drop("first_name","last_name")
display(df)



In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@tjdatabricksete.dfs.core.windows.net/customer")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricksete_cat.silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksete_cat.silver.customers_silver
USING DELTA
LOCATION 'abfss://silver@tjdatabricksete.dfs.core.windows.net/customer'
    


In [0]:
%sql
select * from databricksete_cat.silver.customers_silver